<a href="https://colab.research.google.com/github/mk-16-08/heyjatinnn-gdg-gurugram-build-with-ai-2026/blob/main/multi_agent_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building AI Teams: Multi-Agent Systems with Gemini

**Build with AI by [GDG Gurugram](https://www.linkedin.com/company/gdggurugram) | May 9, 2026**

Welcome! In the next 45 minutes, you will build a working **multi-agent AI system** using Gemini. Not a chatbot. A team of four AI agents that collaborate, **argue**, and produce a research report on any topic you give them.

By the end of this notebook, you will have:

- a **Planner** agent that breaks down a topic into research questions
- a **Researcher** agent that searches the live web using Gemini's Google Search grounding
- a **Writer** agent that drafts the report
- a **Critic** agent that reviews the draft and sends it back if it's not good enough
- an **orchestrator** that wires the team together with a reflection loop
- a **shareable web app** of your agent team — with a public URL you can send to anyone

You can keep using this for free after the workshop.

---

### How to use this notebook

1. Click **File → Save a copy in Drive** (top-left). Work on your copy, not the original.
2. Read each markdown cell, then run the code cell below it (Shift + Enter).
3. **The system prompts are blank `# TODO` for you to write.** That's the part that makes the agents *yours*. The trainer will explain on the slide what each prompt should contain — write it in your own words.
4. If something breaks, check the **Common errors** section.
5. The trainer will pause at checkpoints. If you finish a section early, try the **"Try this"** challenge.

Let's go.


---

## Step 0: Setup

We need three things to start:
1. The Gemini Python SDK installed
2. A free API key from Google AI Studio
3. The key loaded into Colab

### 0.1 Install the SDK

Run the cell below.

> ⚠️ **You will see a red `ERROR: pip's dependency resolver...` warning about `google-auth`. Ignore it.** It's a known Colab quirk where Google's pre-installed packages disagree about a sub-dependency version. The install works fine and nothing in this workshop is affected. If the next cell (`from google import genai`) runs, you're good.


In [1]:
# Pinning google-genai to v1.x because Colab's pre-installed google-cloud-aiplatform expects google-genai<2.0.
# v1.x has every feature we need (Google Search grounding, the new Client API, all of it).
# Gradio is for the final web app step.
!pip install -q "google-genai>=1.66.0,<2.0.0" gradio

### 0.2 Get a free Gemini API key

1. Open [aistudio.google.com/apikey](https://aistudio.google.com/apikey) in a new tab
2. Sign in with your Google account
3. Click **Create API key** → **Create API key in new project**
4. Copy the key. It looks like `AIza...`

The free tier is enough for this whole workshop.

### 0.3 Add the key to Colab securely

In Colab, click the **🔑 key icon** in the left sidebar.

1. Click **Add new secret**
2. Name: `GEMINI_API_KEY` (exactly this, case-sensitive)
3. Value: paste your key
4. Toggle **Notebook access** ON

Then run the cell below.


In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("API key loaded:", os.environ["GEMINI_API_KEY"][:8] + "..." + os.environ["GEMINI_API_KEY"][-4:])


### 0.4 Hello, Gemini

Quick smoke test.


In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hi to a workshop full of developers in one short sentence."
)
print(response.text)


**Expected output:** a one-sentence friendly hello.

#### Common errors

| Error | Fix |
|---|---|
| `SecretNotFoundError` | Secret name is wrong. Must be exactly `GEMINI_API_KEY` and **Notebook access** must be ON. |
| `PermissionDenied: API key not valid` | Re-copy the key from AI Studio. Trailing space is the usual culprit. |
| `ResourceExhausted` / `429` | Free-tier rate limit. Wait 30 seconds and re-run. |

✋ **Checkpoint.** Wait for the trainer.


---

## Step 1: What is an "agent", really?

> **An agent is just an LLM with a clearly defined role and a clear input/output contract.**

That's the whole secret. Every agent we build today is a Python function that:
1. Takes some input
2. Calls Gemini with a focused **system instruction** (its "role")
3. Returns a structured output

A **multi-agent system** is just multiple such functions, where the output of one becomes the input of the next, plus optional decision logic about *which* agent to call when.

We are NOT using LangChain, CrewAI, or any agent framework. We are writing the orchestration ourselves so you can see exactly how it works.

> 🎯 **The most important skill in agentic AI is writing good system prompts.** That's the part you'll write yourself today. Each agent's character lives in its system prompt.

Let's build agent #1.


---

## Step 2: The Planner agent

**Role:** Take a user topic. Break it down into 3 sharp research questions.

**Why we need this:** if you ask an LLM "research X" in one shot, you get a vague essay. If you first decompose X into 3 specific questions and answer each one, you get a real report.

### 👉 Your task: write the Planner's system prompt

Look at the slide for what a good Planner prompt should contain. Then write yours below where it says `# TODO`.

There's no single right answer. Your prompt is yours. Bob's prompt and yours will produce different planners. That's the point.


In [ ]:
import json

# 👉 TODO: Write the Planner system prompt below.
# Hints (from the slide):
#  - state the role clearly (e.g. "You are a research planner.")
#  - what's the input? (a topic)
#  - what's the output? (a JSON array of strings)
#  - exact count? (3 questions)
#  - one example showing the shape
#  - what to forbid? (markdown fences, prose around the JSON)
PLANNER_SYSTEM = """

REPLACE THIS WITH YOUR PROMPT.

"""

# Pre-written below. Don't change it.
def planner(topic: str) -> list[str]:
    """Take a topic, return a list of research questions."""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Topic: {topic}",
        config={"system_instruction": PLANNER_SYSTEM}
    )

    text = response.text.strip()
    # Strip markdown fences in case the model wraps the JSON anyway
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())


### Try it


In [ ]:
topic = "what can developers build with Gemini 2.5 Flash in 2026"

questions = planner(topic)

print(f"Topic: {topic}\n")
print("Research questions:")
for i, q in enumerate(questions, 1):
    print(f"  {i}. {q}")


**Expected output:** a list of specific research questions.

If your output is bad (vague questions, wrong count, JSON parsing error) — that's a feedback signal about your prompt, not a bug. Read your prompt again. Be more specific. Re-run.

#### Common errors

| Error | Fix |
|---|---|
| `JSONDecodeError` | Your prompt didn't constrain output to JSON strictly enough. Add "Return ONLY a JSON array, no prose, no markdown fences." |
| Empty list / 1 question / 5 questions | Your prompt didn't lock the count. Add "exactly 3 questions". |
| Vague questions | Your prompt didn't push for specificity. Add "Each question must be specific and answerable with current web research." |

#### Try this (if you finish early)

Try a deliberately vague topic like `"AI"` and see what your planner does. Then improve your prompt to handle vague inputs better.

✋ **Checkpoint.**


---

## Step 3: The Researcher agent (with live Google Search 🔥)

This is the magic step. We give an agent a **tool**: the ability to search the live web through Gemini's built-in Google Search grounding.

**Role:** Take one question. Search the web. Return an answer with sources.

This is where Gemini specifically shines. Google Search grounding is built right into the API — no scraping, no third-party search service, no flaky workarounds.

### 👉 Your task: write the Researcher's system prompt

The slide will show you what makes a good researcher prompt. Write yours below.


In [ ]:
# 👉 TODO: Write the Researcher system prompt below.
# Hints (from the slide):
#  - state the role (e.g. "You are a research analyst.")
#  - what's the input? (one specific question)
#  - what's the output? (a focused 2-3 paragraph answer)
#  - what to push for? (specifics — numbers, names, dates, current info)
#  - what to forbid? (vague hedging like "it depends", inventing facts)
RESEARCHER_SYSTEM = """

REPLACE THIS WITH YOUR PROMPT.

"""

# Pre-written below.
def researcher(question: str) -> dict:
    """Take a question, search the web, return an answer with sources."""

    # The Google Search tool — this is the key part.
    google_search_tool = types.Tool(google_search=types.GoogleSearch())

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=RESEARCHER_SYSTEM,
            tools=[google_search_tool],
        )
    )

    answer = response.text

    # Pull out sources Gemini actually used
    sources = []
    if response.candidates and response.candidates[0].grounding_metadata:
        chunks = response.candidates[0].grounding_metadata.grounding_chunks or []
        for chunk in chunks:
            if chunk.web:
                sources.append({"title": chunk.web.title, "uri": chunk.web.uri})

    return {"question": question, "answer": answer, "sources": sources}


### Try it


In [ ]:
result = researcher(questions[0])

print(f"Q: {result['question']}\n")
print(f"A: {result['answer']}\n")
print(f"Sources ({len(result['sources'])}):")
for s in result['sources'][:5]:
    print(f"  - {s['title']}")


**Expected output:** a 2-3 paragraph answer with current information, plus a list of source titles.

This is the moment people in the room go "wait, it actually searched the web?" Yes. That's what `google_search` as a tool does.

#### Why this matters

Most AI agent tutorials skip tools because they're messy to wire up. But an agent **without tools is just a clever prompt**. The moment you give an agent the ability to *do* something (search, read a file, call an API), you have a real agent.

#### Common errors

| Error | Fix |
|---|---|
| Empty `sources` list | Sometimes Gemini answers from training without searching. Answer is still grounded; not always a problem. |
| Hedgy "it depends" answer | Your prompt didn't forbid hedging. Add "Do not hedge with 'it depends' without naming the actual factors." |
| `429` rate limit | Wait 30 seconds. Free tier is generous but not infinite. |

✋ **Checkpoint.**


---

## Step 4: The Writer agent

**Role:** Take all the research and synthesize a clean, structured report.

**Why a separate agent?** The Researcher's job is to *find facts*. The Writer's job is to *structure prose*. Different skills. Separating them makes each agent better.

### 👉 Your task: write the Writer's system prompt


In [ ]:
# 👉 TODO: Write the Writer system prompt below.
# Hints (from the slide):
#  - state the role (e.g. "You are a technical writer.")
#  - input? (topic + research findings)
#  - output structure? (executive summary, sections, bottom line)
#  - length cap? (e.g. under 400 words)
#  - what to forbid? (inventing facts, padding, vague hedging)
WRITER_SYSTEM = """

REPLACE THIS WITH YOUR PROMPT.

"""

# Pre-written below.
def writer(topic: str, research: list[dict], feedback: str = "") -> str:
    """Take topic + research + optional critic feedback, return a markdown report."""

    research_block = "\n\n".join([
        f"Q: {r['question']}\nA: {r['answer']}"
        for r in research
    ])

    prompt = f"Topic: {topic}\n\nResearch findings:\n\n{research_block}"

    # If the critic gave us feedback last round, include it
    if feedback:
        prompt += f"\n\n---\nPrevious draft was rejected by editor. Feedback to address:\n{feedback}\n\nWrite an improved draft addressing the feedback above."

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={"system_instruction": WRITER_SYSTEM}
    )

    return response.text


Notice the Writer takes optional `feedback`. That's how the Critic (next step) will push the Writer to improve. We're already setting up the reflection loop.

✋ **Checkpoint.**


---

## Step 5: The Critic agent — where it stops being a pipeline and becomes a *team* 🔥

So far the agents have just passed work down the line. Now we add an agent that **judges** another agent's work and can send it back.

**Role:** Read the Writer's draft. Either approve it or list 2-3 specific improvements needed.

**This is the most important pattern in production agent systems** — it's called a **reflection loop**. Every serious agentic AI framework has some version of it. You're learning the real thing.

### 👉 Your task: write the Critic's system prompt


In [ ]:
# 👉 TODO: Write the Critic system prompt below.
# Hints (from the slide):
#  - state the role (e.g. "You are a senior editor.")
#  - input? (a draft report)
#  - output? STRICT format — either start with "APPROVE" or list specific feedback
#  - what to look for? (specifics, balance, clarity, no fluff)
#  - critical: tell it not to be too lenient OR too harsh
CRITIC_SYSTEM = """

REPLACE THIS WITH YOUR PROMPT.

"""

# Pre-written below.
def critic(report: str) -> dict:
    """Review a draft. Return {'approved': bool, 'feedback': str}."""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Draft to review:\n\n{report}",
        config={"system_instruction": CRITIC_SYSTEM}
    )

    text = response.text.strip()
    approved = text.upper().startswith("APPROVE")

    return {"approved": approved, "feedback": text}


✋ **Checkpoint.** Don't run the orchestrator yet — we wire it up next.


---

## Step 6: Connect the team — orchestrator with a reflection loop

Now we wire everything together. This is where the multi-agent system comes alive.

```
   topic
     │
     ▼
   Planner ──→ 3 questions
     │
     ▼
   Researcher (×3) ──→ research findings
     │
     ▼
   Writer ──→ draft v1
     │
     ▼
   Critic ──→ approved? ──→ YES → done
     │             │
     │             └── NO → feedback ──→ Writer rewrites
     │                                       │
     │                                       └─→ back to Critic (max 3 rounds)
     ▼
  final report
```

The orchestrator below is **fully pre-written**. Read it, run it, watch the team argue.


In [ ]:
def research_agent_team(topic: str, max_rounds: int = 3) -> str:
    """Full pipeline: planner → researcher (×3) → writer ⇄ critic loop."""

    print(f"📋 Topic: {topic}\n")
    print("=" * 60)

    # Step 1: Planner
    print("🧠 PLANNER → breaking down the topic...")
    questions = planner(topic)
    for i, q in enumerate(questions, 1):
        print(f"   {i}. {q}")
    print()

    # Step 2: Researcher
    print("🔎 RESEARCHER → searching the live web...")
    research = []
    for i, q in enumerate(questions, 1):
        print(f"   ({i}/{len(questions)}) {q[:70]}...")
        result = researcher(q)
        research.append(result)
        print(f"       ✓ found {len(result['sources'])} sources")
    print()

    # Step 3 + 4: Writer ⇄ Critic loop
    feedback = ""
    report = ""
    for round_num in range(1, max_rounds + 1):
        print(f"✍️  WRITER → drafting v{round_num}...")
        report = writer(topic, research, feedback=feedback)
        word_count = len(report.split())
        print(f"       ✓ {word_count} words")
        print()

        print(f"🧐 CRITIC → reviewing draft v{round_num}...")
        review = critic(report)

        if review["approved"]:
            print(f"       ✅ APPROVED")
            print()
            break
        else:
            print(f"       ❌ rejected. feedback:")
            for line in review["feedback"].split("\n"):
                if line.strip():
                    print(f"           {line.strip()}")
            print()
            feedback = review["feedback"]

            if round_num == max_rounds:
                print(f"       ⚠️  hit max rounds ({max_rounds}). Shipping last draft anyway.")
                print()

    print("=" * 60)
    print("📄 FINAL REPORT")
    print("=" * 60)
    return report


### Run the full team

This is your moment. Pick a topic and watch your agents collaborate.

⏱️ **Heads up:** this takes 60-90 seconds because the Researcher runs 3 times and the Critic may bounce drafts back. Watch the live console output — you'll see the agents talking.


In [ ]:
# 👉 TODO: Pick your own topic. Make it something you actually care about.
topic = "what can developers build with Gemini 2.5 Flash in 2026"

report = research_agent_team(topic)
print(report)


You just ran a 4-agent team that:

- planned its own work ✅
- searched the live web ✅
- drafted a report ✅
- **judged its own draft and rewrote it until it was good enough** ✅

That last bullet is the difference between a chatbot and an agent system.

#### Try this

Re-run with these and watch how the Critic loop behaves differently:

- something the agents will get right on the first try → Critic approves v1 fast
- something contested or technical → Critic likely bounces v1, you watch v2 improve

✋ **Checkpoint.**


---

## Step 7: Ship it — wrap your agents in a real web app 🚀

Right now your agents only work for you, in this notebook. Let's give them a **public web URL** anyone can use. We'll use **Gradio**, which builds a web UI in 3 lines of code and gives you a free shareable link.

The cell below is pre-written. Run it.


In [ ]:
import gradio as gr

def run_team(topic: str) -> str:
    """Wrapper for Gradio — runs the agent team and returns the final report."""
    if not topic.strip():
        return "Please enter a topic."
    return research_agent_team(topic)

demo = gr.Interface(
    fn=run_team,
    inputs=gr.Textbox(label="Topic", placeholder="e.g. what can developers build with Gemini 2.5 Flash in 2026"),
    outputs=gr.Markdown(label="Report"),
    title="🤖 My AI Research Team",
    description="A 4-agent system (Planner, Researcher, Writer, Critic) built with Gemini at Build with AI by GDG Gurugram.",
    allow_flagging="never"
)

demo.launch(share=True)


When the cell finishes, look for a line like:

```
Running on public URL: https://abc123.gradio.live
```

**That's your app.** Copy that URL. Send it to a friend right now. They can use *your* agent team, with *your* prompts, on any topic.

You didn't run a notebook today. You shipped a web app.

> ⚠️ **Note:** the public URL is free for 72 hours. To keep your app running long-term, deploy to **[Hugging Face Spaces](https://huggingface.co/spaces)** (also free). Same code, permanent URL.

---

## What you built today

- ✅ A 4-agent AI system (not 1, not 3 — *four*, with reflection)
- ✅ Live web research via Gemini's Google Search tool
- ✅ A real reflection loop where agents judge and improve each other's work
- ✅ A working web app with a public URL
- ✅ Zero frameworks, zero magic — just Python + Gemini
- ✅ Free to run, free to share

---

## Where to go from here

- **Add a 5th agent** — e.g., a fact-checker that runs after the Critic approves
- **Add memory** — save research to a JSON file, load relevant past research on new topics
- **Add more tools** — `types.Tool(url_context=types.UrlContext())` lets the researcher read specific webpages, not just search
- **Move to a framework** — once you understand the pattern, [Google ADK](https://google.github.io/adk-docs/) or [LangGraph](https://langchain-ai.github.io/langgraph/) give you orchestration, state, and tracing for free. You wrote it from scratch first, so you'll *understand* what they're doing.
- **Deploy permanently** — Hugging Face Spaces, Render, or Cloud Run. All have free tiers.

---

## Stay in touch

Built by **Jatin** for [GDG Gurugram](https://www.linkedin.com/company/gdggurugram).

If you build something cool with this, tag me — I want to see it.

- Instagram: [@heyjatinnn](https://instagram.com/heyjatinnn)
- YouTube: [@heyjatinnn](https://youtube.com/@heyjatinnn)
- LinkedIn: [@heyjatinnn](https://linkedin.com/in/heyjatinnn)

Tag your build with **#BuildWithAI** and **@gdggurugram** so the rest of the community sees it.

🚀
